# Baghewala Field Digital Twin — Rod Floating & Failure Risk ML
**Objective:** Solve the core Baghewala mechanical problem:
1. **Rod Floating Prediction Model** — Downstroke viscous drag exceeds effective rod weight ($F_{	ext{drag}} > W_{	ext{buoyant}}$)
2. **Equipment Reliability & Fatigue Parting Classifier** — API RP 11L modified Goodman stress ratio & cyclic load range ($\Delta W = PPRL - MPRL$)

Artifacts saved directly to `models/rod_floating/` and `models/failure/`.


In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

MODELS_DIR = Path("models")
ext_path = Path("data/external/baghewala_css_engineered_master.parquet")
df = pd.read_parquet(ext_path)
print(f"Loaded {len(df)} records from {ext_path}")


### 1. Train Rod Floating Risk Model
Inputs: `rod_load_lb`, `spm`, `stroke_length_in`, `vfd_frequency_hz`, `pump_efficiency_fraction`, `end_production_temperature_c`, `estimated_viscosity_at_production_cp`

In [ ]:
mech_features = [
    "css_cycle", "spm", "stroke_length_in", "vfd_frequency_hz",
    "rod_load_lb", "pump_efficiency_fraction", "end_production_temperature_c",
    "estimated_viscosity_at_production_cp", "oil_rate_bopd", "water_cut_fraction"
]

X = df[mech_features].fillna(0)
y_float = df["rod_floating_risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y_float, test_size=0.2, random_state=42)

model_float = GradientBoostingRegressor(n_estimators=120, max_depth=6, learning_rate=0.08, random_state=42)
model_float.fit(X_train, y_train)

y_pred_float = model_float.predict(X_test)
print(f"Rod Floating Model Evaluation:")
print(f"  R² Score: {r2_score(y_test, y_pred_float):.4f}")
print(f"  MAE:      {mean_absolute_error(y_test, y_pred_float):.4f}")

float_dir = MODELS_DIR / "rod_floating"
float_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model_float, float_dir / "rod_floating_gb_v1.joblib")
print(f"Saved artifact to {float_dir / 'rod_floating_gb_v1.joblib'}")


### 2. Train Equipment Failure Risk Model

In [ ]:
y_fail = df["rod_failure_risk"]
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X, y_fail, test_size=0.2, random_state=42)

model_fail = GradientBoostingRegressor(n_estimators=120, max_depth=6, learning_rate=0.08, random_state=42)
model_fail.fit(X_train_f, y_train_f)
y_pred_fail = model_fail.predict(X_test_f)

print(f"Failure Risk Model Evaluation:")
print(f"  R² Score: {r2_score(y_test_f, y_pred_fail):.4f}")
print(f"  MAE:      {mean_absolute_error(y_test_f, y_pred_fail):.4f}")

fail_dir = MODELS_DIR / "failure"
fail_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model_fail, fail_dir / "failure_risk_gb_v1.joblib")
print(f"Saved artifact to {fail_dir / 'failure_risk_gb_v1.joblib'}")
